# NBA MVP Project — Web Scraping

Scrapes the MVP finalists and the 10 player-statistics features used in the CS229 reference paper.

**Features:** G, MP, FG%, 3P%, FT%, TRB, AST, STL, BLK, PTS.

In [9]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)


In [10]:
# 16 historical seasons used for training, plus 2021 for prediction/validation.
training_years = list(range(2005, 2021))
prediction_year = 2021
years = training_years + [prediction_year]

print('Training years:', training_years)
print('Prediction year:', prediction_year)


Training years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Prediction year: 2021


In [11]:
def scrape_top_mvp_candidates(year):
    url = f'https://www.basketball-reference.com/awards/awards_{year}.html'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    response.raise_for_status()

    tables = pd.read_html(response.text)
    mvp_df = None

    for table in tables:
        cols = set(table.columns)
        if {'Player', 'Share'}.issubset(cols) and 'Pts Won' in cols:
            mvp_df = table.copy()
            break

    if mvp_df is None:
        raise ValueError(f'MVP table not found for {year}')

    mvp_df = mvp_df.dropna(subset=['Player']).head(3).copy()
    mvp_df['Rank'] = range(1, len(mvp_df) + 1)
    mvp_df['MVP'] = (mvp_df['Rank'] == 1).astype(int)
    mvp_df['Year'] = year

    return mvp_df[['Rank', 'Player', 'Year', 'MVP']]


In [12]:
mvp_candidates = []

for year in years:
    print(f'Scraping MVP candidates for {year}...')
    try:
        mvp_candidates.append(scrape_top_mvp_candidates(year))
        time.sleep(1)
    except Exception as e:
        print(f'Error in {year}: {e}')

mvp_candidates = pd.concat(mvp_candidates, ignore_index=True)
mvp_candidates

Scraping MVP candidates for 2005...
Error in 2005: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/awards/awards_2005.html
Scraping MVP candidates for 2006...
Error in 2006: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/awards/awards_2006.html
Scraping MVP candidates for 2007...
Error in 2007: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/awards/awards_2007.html
Scraping MVP candidates for 2008...
Error in 2008: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/awards/awards_2008.html
Scraping MVP candidates for 2009...
Error in 2009: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/awards/awards_2009.html
Scraping MVP candidates for 2010...
Error in 2010: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/awards/awards_2010.html
Scraping MVP candidates for 2011...
Error in 2011: 403 Client Error: Forbidden for url: https://www.basket

ValueError: No objects to concatenate

In [ ]:
def scrape_player_stats(year):
    url = f'https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    response.raise_for_status()

    tables = pd.read_html(response.text)
    required = {'Player', 'G', 'MP', 'FG%', '3P%', 'FT%', 'TRB', 'AST', 'STL', 'BLK', 'PTS'}
    stats_df = None

    for table in tables:
        if required.issubset(set(table.columns)):
            stats_df = table.copy()
            break

    if stats_df is None:
        raise ValueError(f'Player statistics table not found for {year}')

    stats_df = stats_df[list(required)].copy()
    stats_df['Year'] = year
    stats_df = stats_df[stats_df['Player'].notna() & (stats_df['Player'] != 'Player')]

    return stats_df


In [ ]:
player_stats = []

for year in years:
    print(f'Scraping player statistics for {year}...')
    try:
        player_stats.append(scrape_player_stats(year))
        time.sleep(1)
    except Exception as e:
        print(f'Error in {year}: {e}')

player_stats = pd.concat(player_stats, ignore_index=True)
player_stats.head()

In [ ]:
feature_columns = ['G', 'MP', 'FG%', '3P%', 'FT%', 'TRB', 'AST', 'STL', 'BLK', 'PTS']

final_data = pd.merge(
    mvp_candidates,
    player_stats,
    on=['Player', 'Year'],
    how='left'
)

final_data = final_data[['Year', 'Rank', 'Player'] + feature_columns + ['MVP']]

for col in feature_columns:
    final_data[col] = pd.to_numeric(final_data[col], errors='coerce')

final_data = final_data.dropna(subset=feature_columns).reset_index(drop=True)
final_data

In [ ]:
final_data.to_csv('NBA_MVP_10_Features.csv', index=False)
print('Saved: NBA_MVP_10_Features.csv')
print('Shape:', final_data.shape)
